# 01 · The model

Every agent starts with a model call. In this notebook you connect LangChain's `ChatOllama` to the
shared local model, make a first call, and get a structured answer back: ShipFast's triage, which sorts a
customer message into an intent and an order ID.

**You will learn**

- how the shared model is set up, and why its settings are fixed
- how to call a chat model with plain text and with messages
- how to get a typed, validated answer instead of free text
- where the token counts and timings of a call live

**Before you start:** Ollama is running (`ollama serve`) and the shared model exists
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`). See the README.

In [1]:
# Check that Ollama is running and the shared model exists. If not, this stops with the command that fixes it.
from importlib.metadata import version

from agent_contracts.harness import load_settings, verify_model

verify_model()
settings = load_settings()  # the model name and Ollama address: the defaults, or your environment's overrides

print("Model:           ", settings.model)
print("LangGraph:       ", version("langgraph"))
print("langchain-ollama:", version("langchain-ollama"))

Model:            qwen3-agents
LangGraph:        1.2.19
langchain-ollama: 1.1.0


## The shared model

All three tutorials (LangGraph, CrewAI and Strands) use the same model with the same settings, so any
difference you see later comes from the framework, not the model:

| Setting | Value | Why |
|---|---|---|
| model | `qwen3-agents` | `qwen3:8b`, a small model that runs on a laptop, with the settings below built in |
| `temperature` | 0 | the most likely answer every time |
| `seed` | 42 | makes repeated runs repeatable |
| `num_ctx` | 16384 | the model's working memory in tokens; Ollama's default of 4096 is too small for agents |
| thinking | off | Qwen 3 can reason before answering; off is 8 to 10 times faster on this machine |

The settings are written into the model itself (`contracts/ollama/Modelfile`) because one framework,
CrewAI, cannot send `num_ctx` per request. LangChain can, so `make_llm()` sends them as well. It lives in
`lg_agents/llm.py`, the one place this tutorial creates its model, so the notebooks and the finished
agents use exactly the same thing.

Here is every setting it passes to `ChatOllama`:

```python
def make_llm(settings: ModelSettings | None = None) -> ChatOllama:
    # The shared settings (or the ones passed in). MODEL and OLLAMA_HOST can override the first two.
    s = settings or load_settings()
    return ChatOllama(
        model=s.model,  # "qwen3-agents" unless MODEL says otherwise
        base_url=s.host,  # where Ollama listens, http://localhost:11434 by default
        temperature=s.temperature,  # 0: always the likeliest next token, so runs are close to repeatable
        seed=s.seed,  # a fixed seed (42), for the same reason
        num_ctx=s.num_ctx,  # the context window in tokens (16,384), sent with every request
        reasoning=s.think,  # False: thinking off (Ollama's `think`)
    )
```

In [ ]:
# Build the model the way the agents do.
from lg_agents.llm import make_llm

llm = make_llm()  # the chat model used in the rest of this notebook

## A first call

`invoke` takes a string (or a list of messages) and returns an `AIMessage`. Ask about something the model
cannot know: ShipFast is a made-up shop, and its policy is not in the model's training data.

In [3]:
# Ask something the model cannot know: ShipFast is a made-up shop.
response = llm.invoke("In one sentence: what is ShipFast's refund policy for lost packages?")

print("Model's answer:", response.content)

Model's answer: ShipFast's refund policy for lost packages typically involves a full refund of the shipping fees once the package is confirmed lost, provided the customer submits proof of the lost package and follows the company's claim process.


The model either says it does not know or makes something up. Both are the point of the next
notebook: a model on its own only knows what it was trained on, and tools are how it reaches real data.

## Messages and roles

A conversation is a list of messages. The **system** message sets the rules; **human** messages are the
user; the model's replies come back as **AI** messages.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    # The system message sets the rules for the whole conversation.
    SystemMessage("You are ShipFast's support assistant. Reply in one short sentence. "
                  "You cannot look anything up yet, so never invent order details."),
    # The human message is what the customer typed.
    HumanMessage("Hi, where is my order A1001?"),
]
reply = llm.invoke(messages)

print("Reply type:", type(reply).__name__)  # an AIMessage: the model's turn in the conversation
print("Reply text:", reply.content)

Reply type: AIMessage
Reply text: Your order A1001 is currently being processed and will be shipped soon.


If that reply states anything about order A1001, such as a status or a delivery date, the model
invented it: it has no access to orders. A rule in the system message makes invention less likely, but
it is not a guarantee, especially with a small model. That is why ShipFast checks facts with tools
(notebook 02) and checks every refund against a ledger in code, never against the model's word.

## Structured output: triage

Free text is fine for a person, but code needs fields it can check. ShipFast's first step is **triage**:
read the customer's message and return an `intent` and an `order_id`. The shape is the shared `Triage`
model from `agent_contracts`, the same class the CrewAI and Strands builds use.

First, look at the shape itself: the fields the model must fill in.

In [5]:
# Triage is a Pydantic model shared by all three tutorials (contracts/agent_contracts/shipfast/schema.py).
from agent_contracts.shipfast.schema import Triage

# Its JSON schema is what the model gets to see. Print each field with its allowed values and description.
for name, field in Triage.model_json_schema()["properties"].items():
    allowed = field.get("enum") or [option["type"] for option in field["anyOf"]]
    print(name)
    print("  allowed:    ", ", ".join(allowed))
    print("  description:", field["description"])

intent
  allowed:     order_status, refund_request, complaint, other
  description: order_status, refund_request, complaint, or other (out of scope)
order_id
  allowed:     string, null
  description: The order ID, such as A1001, or null if there is none


Two fields, and `intent` can only be one of four values. `with_structured_output` sends this schema to
Ollama, which constrains the model's output to valid JSON for it, and LangChain parses the JSON into a
`Triage` object. The prompt explains what each intent means.

In [6]:
# The instructions for triage: what each intent means, and when order_id should be null.
TRIAGE_PROMPT = (
    "Classify the customer's latest message for ShipFast support.\n"
    "intent: order_status (where is my order, when will it arrive), refund_request (an explicit ask for a "
    "refund, return or cancellation), complaint (something is wrong, with no explicit refund ask), or other "
    "(anything that is not about orders, deliveries, refunds or complaints).\n"
    "order_id: the order ID such as A1001 if the message gives one, else null."
)

# with_structured_output wraps the model so that invoke returns a Triage object instead of a message.
triage = llm.with_structured_output(Triage)

result = triage.invoke([
    SystemMessage(TRIAGE_PROMPT),
    HumanMessage("My order A1002 never arrived. I want a refund."),
])

print("Returned:", type(result).__name__)
print("intent:  ", result.intent)
print("order_id:", result.order_id)

Returned: Triage
intent:   refund_request
order_id: A1002


Now run it over the opening message of every ShipFast scenario and compare with the expected answer
from `scenarios.json`. (Some scenarios have a second turn that only makes sense with the first one in
memory; that is notebook 04.)

In [7]:
# Triage the first message of every ShipFast scenario and compare it with the expected answer.
import time

from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
correct = 0

print(f"{'check':6} {'time':>5}  {'intent':15} {'order':6} customer message")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    expected = scenario["expected"]  # the right intent and order ID, from scenarios.json

    start = time.perf_counter()
    got = triage.invoke([SystemMessage(TRIAGE_PROMPT), HumanMessage(text)])
    seconds = time.perf_counter() - start

    ok = (got.intent, got.order_id) == (expected["intent"], expected["order_id"])
    correct += ok
    print(f"{'ok' if ok else 'MISS':6} {seconds:4.1f}s  {got.intent:15} {str(got.order_id):6} {text}")

print(f"\n{correct} of {len(seen)} messages triaged correctly")

check   time  intent          order  customer message


ok      3.9s  order_status    A1001  Hi, where is my order A1001?


ok      4.4s  refund_request  A1002  My order A1002 never arrived and tracking says it's lost. I want a refund.


ok      4.2s  refund_request  A1003  Order A1003 is more than a week late. Please cancel it and refund me.


ok      3.8s  complaint       A1004  My vase set from order A1004 arrived broken.


ok      3.6s  refund_request  A1005  I want to return order A1005, the stand wobbles.


ok      3.6s  order_status    A9999  Where is order A9999?


ok      2.8s  other           None   Can you write me a poem about cats?

7 of 7 messages triaged correctly


## What a call costs

Every `AIMessage` carries the token counts Ollama reported (`usage_metadata`) and its timings
(`response_metadata`, in nanoseconds). Notebook 07 collects these into the shared harness so the three
frameworks are measured the same way. With structured output the parsed object hides the message, so ask
for it with `include_raw=True`.

In [8]:
# include_raw=True returns the raw AIMessage next to the parsed Triage; the message carries the usage numbers.
both = llm.with_structured_output(Triage, include_raw=True).invoke(
    [SystemMessage(TRIAGE_PROMPT), HumanMessage("Where is order A9999?")])
message = both["raw"]
usage = message.usage_metadata  # token counts, as Ollama reported them
seconds = message.response_metadata["total_duration"] / 1e9  # Ollama reports nanoseconds

print("Parsed answer:", both["parsed"])
print("Input tokens: ", usage["input_tokens"], "(instructions, schema and message)")
print("Output tokens:", usage["output_tokens"], "(the model's answer)")
print(f"Time:          {seconds:.2f} s")

Parsed answer: intent='order_status' order_id='A9999'
Input tokens:  121 (instructions, schema and message)
Output tokens: 23 (the model's answer)
Time:          3.94 s


## How the others do this

- **CrewAI** creates the model with `LLM(model="ollama/qwen3-agents", reasoning_effort="none")`. It talks
  to Ollama's OpenAI-style API, which turns thinking off through `reasoning_effort` and ignores a
  per-request `num_ctx`; that is why the settings live in the model.
- **Strands** uses `OllamaModel(host, model_id="qwen3-agents", additional_args={"think": False})`. Like
  LangChain it talks to Ollama's own API and can send every setting per request.

All three get the same `Triage` class from `agent_contracts`.

## Takeaways

- The model is a plain object you call with text or messages; LangChain's `ChatOllama` wraps Ollama.
- Fixing the model's settings is what makes a framework comparison fair.
- Structured output turns a reply into a validated object your code can trust.
- A model alone cannot see ShipFast's orders or policy. Next: **02 · Tools**.